# Run GEE Spatial Extractions

Use this notebook as the Colab runner. The reusable workflow code lives in the repo.

Before running, upload the zipped watershed shapefile listed in `config/gee-assets.yml` to Earth Engine using the asset name in that same config file. The active run is set in `config/run-list.yml`; the notebook builds the full run list and launches only the configured chunk for that session.

The current active run is set in `config/run-list.yml`. The first real annual pull uses the shared 2001-2022 window across ERA5-Land, MODIS, and GLC_FCS30D. It exports to the Drive folder named `Google-Earth-Engine`.

Earth Engine shortens some shapefile column names during upload. This notebook checks for the shortened `run_grp` field before filtering runs.


In [ ]:
REPO_URL = "https://github.com/global-river-chem/data-workflow_spatial.git"
REPO_DIR = "data-workflow_spatial"

import os
import subprocess

if not os.path.isdir(REPO_DIR):
    subprocess.run(["git", "clone", REPO_URL], check=True)

os.chdir(REPO_DIR)

In [ ]:
!pip -q install -r requirements.txt

In [ ]:
from src.gee_spatial.auth import initialize
from src.gee_spatial.extraction import load_run_config

assets = load_run_config("config/gee-assets.yml")
products = load_run_config("config/driver-products.yml")
run_list = load_run_config("config/run-list.yml")

ee = initialize(project=assets.get("earth_engine", {}).get("project"))

In [ ]:
from src.gee_spatial.datasets import continuous_products, product_names
from src.gee_spatial.extraction import load_watersheds
from src.gee_spatial.runs import all_run_groups, choose_property_name

watersheds = load_watersheds(assets["watersheds"]["asset_id"])
configured_run_group_column = assets.get("watersheds", {}).get(
    "earth_engine_run_group_column",
    run_list.get("site_groups", {}).get("column", "run_group"),
)
run_group_column = choose_property_name(
    watersheds,
    configured_run_group_column,
    alternatives=["run_group", "run_grp"],
)
available_run_groups = all_run_groups(watersheds, run_group_column)
print("Products:", product_names(products))
print("Continuous products:", continuous_products(products))
print("Run group column:", run_group_column)
print("Asset rows:", watersheds.size().getInfo())
print("Run groups:", len(available_run_groups), available_run_groups[:10])
print("Run group counts:", watersheds.aggregate_histogram(run_group_column).getInfo())
print(watersheds.limit(1).getInfo())


In [ ]:
from datetime import datetime, timezone

from src.gee_spatial.export import export_table_to_drive, task_summary
from src.gee_spatial.extraction import (
    ERA5_EXPORT_COLUMNS,
    EXPORT_COLUMNS,
    extract_continuous_product,
    extract_era5_land_products,
)
from src.gee_spatial.runs import build_run_list, export_name, filter_watersheds_by_group, run_list_chunk

run_rows = build_run_list(run_list, available_run_groups)
launch_settings = run_list.get("launch", {})
start_at = int(launch_settings.get("start_at", 0))
max_tasks = launch_settings.get("max_tasks_per_session", 5)
rows_to_launch = run_list_chunk(run_rows, start_at=start_at, max_tasks=max_tasks)
active_run = run_list.get("active_run")
run_settings = (run_list.get("runs") or {}).get(active_run, {})
run_export = bool(run_settings.get("launch_export", False))

print("Active run:", active_run)
print("Total planned tasks:", len(run_rows))
print("Tasks in this chunk:", len(rows_to_launch))
print("Launch export:", run_export)

launched_tasks = []

for row_number, run_row in enumerate(rows_to_launch, start=start_at + 1):
    MODE = run_row.get("mode", "single_product")
    PRODUCT = run_row.get("product")
    PRODUCTS = run_row.get("products")
    YEAR = run_row.get("year")
    MONTH = run_row.get("month")
    PERIOD = run_row.get("period")
    RUN_GROUP = run_row.get("run_group")

    selected_watersheds = filter_watersheds_by_group(
        watersheds,
        run_group=RUN_GROUP,
        run_group_column=run_group_column,
    )
    selected_row_count = selected_watersheds.size().getInfo()
    export_label = MODE if MODE == "era5_land" else PRODUCT
    out_name = export_name(export_label, year=YEAR, month=MONTH, run_group=RUN_GROUP, period=PERIOD)

    print("\nTask", row_number, "of", len(run_rows))
    print("Planned export:", out_name)
    print("Selected rows:", selected_row_count)

    if selected_row_count == 0:
        raise ValueError(
            f"No watershed rows selected for {RUN_GROUP} using column {run_group_column}. "
            "Check the uploaded asset column names before exporting."
        )

    if not run_export:
        continue

    if MODE == "era5_land":
        export_rows = extract_era5_land_products(
            products,
            selected_watersheds,
            year=YEAR,
            month=MONTH,
            product_names=PRODUCTS,
        )
        selectors = ERA5_EXPORT_COLUMNS
    else:
        export_rows = extract_continuous_product(
            PRODUCT,
            products,
            selected_watersheds,
            year=YEAR,
            month=MONTH,
        )
        selectors = EXPORT_COLUMNS

    task = export_table_to_drive(
        export_rows,
        description=out_name,
        folder=assets["exports"]["drive_folder"],
        file_name_prefix=out_name,
        selectors=selectors,
    )
    launched_tasks.append({"name": out_name, "task": task, "launched_at": datetime.now(timezone.utc)})
    print(task_summary(task))


In [ ]:
import time
from datetime import datetime, timezone

done_states = {"COMPLETED", "FAILED", "CANCELLED"}
poll_seconds = 60

if not launched_tasks:
    print("No tasks were launched in this session")
else:
    while True:
        now = datetime.now(timezone.utc)
        print("\nTask status at", now.isoformat(timespec="seconds"))
        all_done = True

        for item in launched_tasks:
            status = item["task"].status()
            state = status.get("state")
            elapsed_min = (now - item["launched_at"]).total_seconds() / 60
            print(f"{item['name']}: {state}, elapsed {elapsed_min:.1f} min")

            if state not in done_states:
                all_done = False

        if all_done:
            break

        time.sleep(poll_seconds)
